In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from scipy.cluster.hierarchy import fcluster

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_subject_topk(subject_index=2):
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk[subject_index]

In [ ]:
def load_all_subjects_topk():
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk

In [ ]:
def get_top_k_keys(d, k, reverse=True):
    """
    Returns the top k keys in a dictionary that have the highest values.

    Parameters:
    d (dict): The input dictionary.
    k (int): The number of top keys to return.

    Returns:
    list: A list of the top k keys with the highest values.
    """
    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=reverse)[:k]
    return top_k_keys

# Example usage
d = {'a': 10, 'b': 20, 'c': 15, 'd': 5, 'e': 25}
k = 3
print(get_top_k_keys(d, k))  # Output: ['e', 'b', 'c']

In [ ]:
def get_top_k_keys_all_subjects():
    topk = load_all_subjects_topk()
    topk_keys = {}
    for subject_index in topk.keys():
        topk_keys[subject_index] = get_top_k_keys(topk[subject_index], 60)
    return topk_keys


In [ ]:
top_k_all_subjects = get_top_k_keys_all_subjects()

In [ ]:
def calculate_agreement_top_channels(top1, top2):

    agreement_results = {}

    agreement_count = sum(1 for ch in top1 if ch in top2)
    agreement_ratio = agreement_count / 10.0
            
    agreement_results = agreement_ratio

    return agreement_results


In [ ]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=29):
    data_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/explanation_data"
    file_path = os.path.join(data_dir, f"subject_{subject_index}_results.pkl")

    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f)   
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 10-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]
    #top_k_channels_dict = {ch: sum_over_channel_points_dict[ch] for ch in top_k_channels}
    

    return top_k_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

In [ ]:
def channel_importance(explanations, ch_names):
    # explanation are of shape (trials, channels, timepoints)
    # we want to get the importance of each channel over all trials and timepoints
    # we also want each trial to be weighed equally so we MinMaxscale over timepoint and trial dimensions

    num_trials, num_channels, num_timepoints = explanations.shape
    channel_importance_scores = np.zeros(num_channels)
    channel_importance_dict = {}
    
    # Process each trial individually to ensure equal weighting
    for trial_idx in range(num_trials):
        trial_data = explanations[trial_idx]
        
        # Flatten channel and timepoint dimensions for scaling
        flattened = trial_data.reshape(-1)
        
        # Apply z-score normalization to the trial data
        mean = np.mean(flattened,)
        std = np.std(flattened)
        
        # Handle case where standard deviation is zero

        
        # Normalize the data
        normalized = (flattened - mean) / std
        trial_normalized = normalized.reshape(num_channels, num_timepoints)
        # Average across timepoints to get importance per channel for this trial
        trial_importance = np.mean(trial_normalized, axis=1)
        
        # Add to the cumulative scores
        channel_importance_scores += trial_importance
    
    # Average across all trials to get final importance scores
    channel_importance_scores /= num_trials

    for ch in ch_names:
        channel_importance_dict[ch] = channel_importance_scores[ch_names.index(ch)]

    
    return channel_importance_dict

In [ ]:
_,_, explanations,_ = load_predicted_amplitude_for_subject(subject_index=29)

In [ ]:
#channel_importance(explanations, ch_names)

In [ ]:
mne.set_log_level("ERROR")

In [ ]:
def get_all_subject_importanes():
    all_subject_importances = {}
    for subject_index in top_k_all_subjects.keys():
        _,_, explanations, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index)
        all_subject_importances[subject_index] = channel_importance(explanations, ch_names)
    return all_subject_importances

In [ ]:
all_subject_importances = get_all_subject_importanes()

In [ ]:
cfg = load_config()
top_10_per_subject_abs = {}
all_channels_linear_abs = {}
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_10_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_10_per_subject_abs[subject_index] = top_10_channels
    all_channels_linear_abs[subject_index] = sum_over_channel_points_dict

In [ ]:
all_channels_linear_abs

In [ ]:
cfg = load_config()
top_10_per_subject= {}
all_channels_linear = {}
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_10_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": gradshap}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_10_per_subject[subject_index] = top_10_channels
    all_channels_linear[subject_index] = sum_over_channel_points_dict

In [ ]:
all_channels_linear

In [ ]:
cfg = load_config()
top_10_per_subject_negative= {}

load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_10_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": -gradshap}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_10_per_subject_negative[subject_index] = top_10_channels

In [ ]:
def get_common_channels(top_k_all_subjects):
    # Get the first subject's channel names as a set
    common_channels = set(top_k_all_subjects[1])
    
    # Intersect with each other subject's channels
    for subject_id in top_k_all_subjects.keys():
        subject_channels = set(top_k_all_subjects[subject_id])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(top_k_all_subjects.keys())} subjects")
    print("Common channels:", common_channels)
    
    return common_channels
common_channels = get_common_channels(top_k_all_subjects)

In [ ]:
def create_index_groups(uncertainties, subject_index, group_size=100):

    index_groups_all = {}
    index_groups_subject = []
    
    start = 0
    while start < len(uncertainties):
        end = min(start + group_size, len(uncertainties)-20)
        
        index_group = np.zeros(len(uncertainties), dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    
    index_groups_all[subject_index] = index_groups_subject
    return index_groups_all

In [ ]:
def plot_agreement_topomap(top1, top2, top1_negative, top2_negative, top1_abs, top2_abs, ch_names, info_subj1, agreement, agreement_negative, agreement_abs, subj1, subj2, subfig=None):
    # Create figure based on input

    common_channels = {}
    common_channels_abs = {}

    common_channels= np.intersect1d(top1, top2)
    common_channels_negative = np.intersect1d(top1_negative, top2_negative)
    common_channels_abs = np.intersect1d(top1_abs, top2_abs)

    agree_channels = np.zeros(len(ch_names))
    agree_channels_negative = np.zeros(len(ch_names))
    agree_channels_abs = np.zeros(len(ch_names))
    for ch in common_channels:
        agree_channels[ch_names.index(ch)] = 1
    for ch in common_channels_negative:
        agree_channels_negative[ch_names.index(ch)] = 1
    for ch in common_channels_abs:
        agree_channels_abs[ch_names.index(ch)] = 1
    

    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 3))
    fig.suptitle(f' {subj1} vs {subj2}')
    mne.viz.plot_topomap(agree_channels, info_subj1, show=False, names=ch_names, 
                           axes=axs[0], image_interp="nearest")
    mne.viz.plot_topomap(agree_channels_abs, info_subj1, show=False, names=ch_names,
                           axes=axs[2], image_interp="nearest")
    mne.viz.plot_topomap(agree_channels_negative, info_subj1, show=False, names=ch_names,
                           axes=axs[1], image_interp="nearest")
    axs[0].set_title(f'agreement ratio: {agreement:.2f}')
    axs[1].set_title(f'agreement ratio negative: {agreement_negative:.2f}')
    axs[2].set_title(f'agreement ratio abs: {agreement_abs:.2f}')
  


In [ ]:
def plot_agreement_topomap_importances(top1,top2, ch_names, info_subj1, agreement,subj1, subj2 ):
    common_channels= np.intersect1d(top1, top2)
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(3, 3))
    fig.suptitle(f' {subj1} vs {subj2}')

    agree_channels = np.zeros(len(ch_names))
    for ch in common_channels:
        agree_channels[ch_names.index(ch)] = 1
    mne.viz.plot_topomap(agree_channels, info_subj1, show=False, names=ch_names, 
                           axes=ax, image_interp="nearest")
    ax.set_title(f'agreement ratio: {agreement:.2f}')

    

In [ ]:
def load_subject_info(subject_index):

    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info_subj = epochs.info
    return info_subj

In [ ]:
def load_info_all_subjects():
    all_subjects_info = {}
    for subject_index in cfg.dataset.test_subject_indices:
        _, _, _, ch_names = load_predicted_amplitude_for_subject(subject_index)

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_info[subject_index] = {"ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_info

# clustering functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform

def spatial_channel_similarity(top_channels_i, top_channels_j, common_channels, 
                              channel_dist_matrix_i, channel_dist_matrix_j, sigma=0.1):
    """
    Compute similarity between two subjects' top channels accounting for spatial proximity,
    using subject-specific distance matrices.
    
    Parameters:
    -----------
    top_channels_i, top_channels_j : list
        Lists of channel names for the top channels of subjects i and j
    ch_names : list
        List of all channel names
    channel_dist_matrix_i : np.array
        Distance matrix for subject i
    channel_dist_matrix_j : np.array
        Distance matrix for subject j
    sigma : float
        Parameter controlling how quickly similarity falls off with distance
    
    Returns:
    --------
    float : Similarity score between 0 and 1
    """
    # Convert channel names to indices
    # Right now this computes the channel index in common channels for each of the top 10 channels of a subject
    indices_i = [common_channels.index(ch) for ch in top_channels_i if ch in common_channels]
    indices_j = [common_channels.index(ch) for ch in top_channels_j if ch in common_channels]
    
    

    # Using subject i's distance matrix
    # For each channel in subject i, find the closest match in subject j of the top 10 channelss
    similarities_i_to_j = []
    # for each channel in the top 10 of subect 1, search for the channel in the top 10 of subject 2 that is most similar weighted by a gaussian kernel
    # computes distance to each channel in the top 10 of subject 2
    for idx_i in indices_i:
        dists = [channel_dist_matrix_i[idx_i, idx_j] for idx_j in indices_j]
        similarities = [np.exp(-d**2/sigma) for d in dists]
        # find the most similar channel
        similarities_i_to_j.append(max(similarities))
    
    # Using subject j's distance matrix
    # For each channel in subject j, find the closest match in subject i
    similarities_j_to_i = []
    for idx_j in indices_j:
        dists = [channel_dist_matrix_j[idx_j, idx_i] for idx_i in indices_i]
        # does a gaussian kernel make sense here?
        similarities = [np.exp(-d**2/sigma) for d in dists]
        similarities_j_to_i.append(max(similarities))
    
    # Combine the similarities (average)
    overall_similarity = (sum(similarities_i_to_j) + sum(similarities_j_to_i)) / (len(indices_i) + len(indices_j))
    
    return overall_similarity

def compute_similarity_matrix(top_channels_by_subject, common_channels, channel_dist_matrices, sigma=0.1):
    """
    Compute similarity matrix between all pairs of subjects using subject-specific distance matrices.
    
    Parameters:
    -----------
    top_channels_by_subject : dict
        Dictionary with subject IDs as keys and lists of top channel names as values
    ch_names : list
        List of all channel names
    channel_dist_matrices : dict
        Dictionary with subject IDs as keys and channel distance matrices as values
    sigma : float
        Parameter controlling how quickly similarity falls off with distance
    
    Returns:
    --------
    sim_matrix : np.array
        Similarity matrix
    subjects : list
        List of subject IDs
    """
    subjects = list(top_channels_by_subject.keys())
    n_subjects = len(subjects)
    
    # Initialize similarity matrix
    sim_matrix = np.zeros((n_subjects, n_subjects))
    
    # Compute pairwise similarities
    for i in range(n_subjects):
        subj_i = subjects[i]
        for j in range(i, n_subjects):
            subj_j = subjects[j]
            
            if i == j:
                sim_matrix[i, j] = 1.0  # Self-similarity is 1
            else:
                # Use each subject's own distance matrix
                sim = spatial_channel_similarity(
                    top_channels_by_subject[subj_i],
                    top_channels_by_subject[subj_j],
                    common_channels,
                    channel_dist_matrices[subj_i],
                    channel_dist_matrices[subj_j],
                    sigma
                )
                sim_matrix[i, j] = sim
                sim_matrix[j, i] = sim  # Similarity is symmetric
    
    return sim_matrix, subjects

def visualize_similarity_matrix(sim_matrix, subjects):
    """
    Visualize the similarity matrix as a heatmap.
    """
    plt.figure(figsize=(12, 10))
    plt.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Similarity')
    plt.xticks(range(len(subjects)), subjects, rotation=90)
    plt.yticks(range(len(subjects)), subjects)
    plt.title('Subject Similarity Based on Top EEG Channels (Spatially Weighted)')
    plt.tight_layout()
    plt.show()

def cluster_subjects(top_channels_by_subject, common_channels, channel_dist_matrices, 
                    sigma=0.1, method='average', ranking_method="normal"):
    """
    Cluster subjects based on similarity of top channels with spatial weighting.
    
    Parameters:
    -----------
    top_channels_by_subject : dict
        Dictionary mapping subject IDs to lists of their top channels
    ch_names : list
        List of all channel names
    channel_dist_matrices : dict
        Dictionary mapping subject IDs to their channel distance matrices
    sigma : float
        Parameter controlling how quickly similarity falls off with distance
    method : str
        Linkage method for hierarchical clustering

    """
    # Compute similarity matrix
    sim_matrix, subjects = compute_similarity_matrix(
        top_channels_by_subject, common_channels, channel_dist_matrices, sigma
    )
    
    # Visualize similarity matrix
    title_suffix = ranking_method
    
        
    plt.figure(figsize=(12, 4))
    plt.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Similarity')
    plt.xticks(range(len(subjects)), subjects, rotation=90)
    plt.yticks(range(len(subjects)), subjects)
    plt.title(f'Subject Similarity Based on Explanation function {title_suffix}')
    plt.tight_layout()
    plt.show()
    
    # Convert similarity to distance for clustering
    dist_matrix = 1 - sim_matrix
    
    # Perform hierarchical clustering
    linkage_matrix = linkage(squareform(dist_matrix), method=method)
    
    # Plot dendrogram
    plt.figure(figsize=(12, 4))
    dendrogram(
        linkage_matrix,
        labels=subjects,
        leaf_font_size=10
    )
    plt.title(f'Hierarchical Clustering of Subjects{title_suffix}')
    plt.xlabel('Distance')
    plt.tight_layout()
    plt.show()
    
    return linkage_matrix, sim_matrix, subjects

def create_distance_matrices(all_subjects_info, common_channels):
    """
    Create distance matrices for each subject based on channel coordinates.
    
    Parameters:
    -----------
    all_subjects_data : dict
        Dictionary containing subject data including info_subj for each subject
        
    Returns:
    --------
    dict : Dictionary mapping subject IDs to their channel distance matrices
    """
    channel_dist_matrices = {}
    
    for subject_id, subject_data in all_subjects_info.items():
        info = subject_data["info_subj"]
        ch_names = subject_data["ch_names"]
        
        # Extract channel coordinates
        channel_coordinates = {}
        for ch in common_channels:
            try:
                # Get the 3D coordinates (first 3 elements of loc)
                idx = ch_names.index(ch)
                channel_coordinates[ch] = info['chs'][idx]["loc"][:3]
            except (KeyError, IndexError):
                print(f"Warning: Could not find coordinates for channel {ch} in subject {subject_id}")
                # Use zeros as fallback
                channel_coordinates[ch] = np.zeros(3)
        
        # Create array of channel locations
        channel_locations = np.array([channel_coordinates[ch] for ch in common_channels])
        
        # Compute distance matrix
        channel_dist_matrices[subject_id] = squareform(pdist(channel_locations))
    
    return channel_dist_matrices

def analyze_explanation_function(top_channels_all_subjects, all_subjects_info, common_channels, sigma=0.1, ranking_method="normal"):
    """
    Analyze clustering for a specific frequency band and amplification factor.
    
    Parameters:
    -----------
    top_channels_median_all_subjects : dict
        Nested dictionary with structure [subject_id][band_name][amp_factor] = list of channels
    all_subjects_data : dict
        Dictionary containing subject data
    sigma : float
        Parameter controlling spatial similarity weighting
    """

    # Create distance matrices
    channel_dist_matrices = create_distance_matrices(all_subjects_info, common_channels)
    
    # Perform clustering
    linkage_matrix, sim_matrix, subjects = cluster_subjects(
        top_channels_all_subjects, 
        common_channels, 
        channel_dist_matrices, 
        sigma=sigma,
        method='average',
        ranking_method=ranking_method

    )
    
    return linkage_matrix, sim_matrix, subjects




In [ ]:
def get_common_channels(all_subjects_data):
    # Get the first subject's channel names as a set
    first_subject = list(all_subjects_data.keys())[0]
    common_channels = set(all_subjects_data[first_subject]["ch_names"])
    
    # Intersect with each other subject's channels
    for subject_id in all_subjects_data.keys():
        subject_channels = set(all_subjects_data[subject_id]["ch_names"])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(all_subjects_data)} subjects")
    print("Common channels:", common_channels)
    
    return common_channels

In [ ]:
all_subjects_info = load_info_all_subjects()

In [ ]:
def filter_top_channels_to_common(top_channels_all_subjects, common_channels):
    """
    Filter top channels for all subjects to only include channels that are common across all subjects.
    
    Parameters:
    -----------
    top_channels_median_all_subjects : dict
        Dictionary containing top channels for each subject
    common_channels : list
        List of channel names that are common across all subjects
        
    Returns:
    --------
    dict : Filtered dictionary with only common channels
    """
    filtered_top_channels = {}
    
    # Iterate through each subject
    for subject_id in top_channels_all_subjects:
        filtered_top_channels[subject_id] = {}
        
        # Filter to only include common channels
        filtered_channels = [ch for ch in top_channels_all_subjects if ch in common_channels]
        filtered_top_channels[subject_id] = filtered_channels
    
    return filtered_top_channels

# Filter the top channels
#filtered_top_channels_median_all_subjects = filter_top_channels_to_common(
#    top_channels_median_all_subjects, 
#    common_channels

In [ ]:
def get_cluster_labels(linkage_matrix, n_clusters=4):
    """Extract cluster labels from linkage matrix"""

    return fcluster(linkage_matrix, n_clusters, criterion='maxclust')

In [ ]:
import itertools
def plot_cluster_topoplots(linkage_matrix, top_all_subjects, subjects, all_subjects_info, common_channels, n_clusters=5, label="positive"):
    """
    Plot topographic maps of cluster centroids for each cluster.
    
    Parameters:
    -----------
    linkage_matrix : np.array
        Linkage matrix from hierarchical clustering
    subjects : list
        List of subject IDs
    all_subjects_data : dict
        Dictionary containing subject data
    common_channels : list
        List of common channel names
    n_clusters : int
        Number of clusters to extract
    """
    # Extract cluster labels
    cluster_labels = get_cluster_labels(linkage_matrix, n_clusters)
    np.save(f"cluster_labels_top_k_linear_{label}.npy", cluster_labels)
    
    # Create figure
    fig, axs = plt.subplots(1, n_clusters, figsize=(15, 5))
    ch_names = all_subjects_info[1]["ch_names"]
    # Iterate through each cluster
    for cluster_id in range(1, n_clusters + 1):
        # Find subjects in this cluster
        cluster_subjects = [subj for subj, label in zip(subjects, cluster_labels) if label == cluster_id]
        
        # Average the top channels for these subjects
        cluster_top_channels = []
        all_agree_channels_cluster = np.zeros(len(ch_names))
        for subject_id1, subject_id2 in itertools.combinations(cluster_subjects,2):
            #cluster_top_channels.extend(filtered_top_channels_median_all_subjects[subject_id1][band_name][factor])
            pair_agreement = np.intersect1d(top_all_subjects[subject_id1], 
                                                  top_all_subjects[subject_id2])
            pair_agreement = [ch for ch in pair_agreement if ch in common_channels]
            for ch in pair_agreement:
                all_agree_channels_cluster[ch_names.index(ch)] += 1

            # find all channels that are in the top 
            #for band_name in filtered_top_channels_median_all_subjects[subject_id]:
            #    cluster_top_channels.extend(filtered_top_channels_median_all_subjects[subject_id][band_name][factor])
        # Compute average topographic map
        #cluster_topo = np.zeros(len(ch_names))

        # from ch_names of a subject find indices of common_channels+
        # then set the corresponding indices to 
        #for ch in cluster_top_channels:
            #cluster_topo[ch_names.index(ch)] = 1



        # Plot topomap
        mne.viz.plot_topomap(all_agree_channels_cluster, all_subjects_info[1]["info_subj"], show=False, names=ch_names, axes=axs[cluster_id - 1])
        axs[cluster_id - 1].set_title(f'Cluster {cluster_id}, N: {len(cluster_subjects)}', fontsize=18)
    
    plt.suptitle('Average topomap of clusters', fontsize=20)
    plt.tight_layout()
    fig.savefig(f"linear_top_k_topomap_{label}.png")
    plt.show()

In [ ]:
common_channels = get_common_channels(all_subjects_info)

In [ ]:
channel_dist_matrices = create_distance_matrices(all_subjects_info, common_channels)

# usual top 10 agreement 

## plot pairwise topomaps

In [ ]:
import itertools
cfg = load_config()
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    top1_abs = top_10_per_subject_abs[subject_index1]
    top2_abs = top_10_per_subject_abs[subject_index2]
    top1 = top_10_per_subject[subject_index1]
    top2 = top_10_per_subject[subject_index2]
    top1_negative = top_10_per_subject_negative[subject_index1]
    top2_negative = top_10_per_subject_negative[subject_index]
    info_subj1 = load_subject_info(subject_index1)

    agreement = calculate_agreement_top_channels(top1, top2)
    agreement_abs = calculate_agreement_top_channels(top1_abs, top2_abs)
    agreement_negative = calculate_agreement_top_channels(top1_negative, top2_negative)
    #print(agreement)
    plot_agreement_topomap(top1, top2, top1_negative, top2_negative, top1_abs, top2_abs, info_subj1.ch_names, info_subj1, agreement, agreement_negative, agreement_abs, subject_index1, subject_index2)


## cluster explanation function usual approach

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_explanation_function(top_10_per_subject, all_subjects_info, common_channels, sigma=0.0025, ranking_method="normal")

In [ ]:
plot_cluster_topoplots(linkage_matrix, top_10_per_subject, subjects, all_subjects_info, common_channels, n_clusters=5, label="normal")

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_explanation_function(top_10_per_subject_negative, all_subjects_info, common_channels, sigma=0.0025, ranking_method="normal")

In [ ]:
plot_cluster_topoplots(linkage_matrix, top_10_per_subject_negative, subjects, all_subjects_info, common_channels, n_clusters=5, label="negative")

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_explanation_function(top_10_per_subject_abs, all_subjects_info, common_channels, sigma=0.0025, ranking_method="normal")

In [ ]:
plot_cluster_topoplots(linkage_matrix, top_10_per_subject_abs, subjects, all_subjects_info, common_channels, n_clusters=5, label="abs")

## rank correlations

In [ ]:
import scipy
def calculate_pairwise_correlations(importances1, importances2, common_channels):

    correlation_results = {}

        # Extract the top 10 channels for both subjects

            # Extract values for common channels only
    common_channel_values1 = np.array([importances1[ch] for ch in common_channels])
    common_channel_values2 = np.array([importances2[ch] for ch in common_channels])
            
    # Calculate correlations
    pearson_corr = np.corrcoef(common_channel_values1, common_channel_values2)[0,1]
    spearman_corr = scipy.stats.spearmanr(common_channel_values1, common_channel_values2).correlation
            
    # Store results
    correlation_results= {
                'pearson': pearson_corr,
                'spearman': spearman_corr}
  
    return correlation_results

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, take_abs=False, label="positive"):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman'])

        else:

            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman']
        

    # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    np.save(f"distance_matrices/top_k_linear_{label}_pearson.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/top_k_linear_{label}_spearman.npy", distance_matrix_spearman)


    
    fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

    axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
    axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

    #cbar = fig.colorbar(im_spearman, ax=axs.ravel().tolist(), location='bottom', shrink=0.35)
    #cbar.set_label('Correlation')

    # Set axis labels
    for ax in axs:
        ax.set_xticks(np.arange(len(subject_indices)))
        ax.set_yticks(np.arange(len(subject_indices)))
        ax.set_xticklabels(subject_indices)
        ax.set_yticklabels(subject_indices)
        ax.set_xlabel('Subject indices')
        ax.set_ylabel('Subject indices')
        plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
    
    axs[0].set_title('Pearson correlation')
    axs[1].set_title('Spearman correlation')

    # Add text annotations
    for i in range(len(subject_indices)):
        for j in range(len(subject_indices)):
            axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                       ha="center", va="center", color="white")
            axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                       ha="center", va="center", color="white")



                                           

In [ ]:
import itertools

In [ ]:
rank_correlations_all_pairs_abs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    rank_correlations_all_pairs_abs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(all_channels_linear_abs[subject_index1], all_channels_linear_abs[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs_abs, take_abs=False, label="abs")

In [ ]:
rank_correlations_all_pairs= {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    rank_correlations_all_pairs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(all_channels_linear[subject_index1], all_channels_linear[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs, take_abs=False)

# all importances

## paris of subjects

In [ ]:
all_subject_importances = get_all_subject_importanes()

In [ ]:
np.save("normalized_importances.npy", all_subject_importances, allow_pickle=True)

In [ ]:
all_subject_importances_abs = {}
for subject_index, importance in all_subject_importances.items():
    all_subject_importances_abs[subject_index] = {ch: abs(importance[ch]) for ch in importance.keys()}

In [ ]:
top_10_importances_all_subjects = {}
top_10_importances_all_subjects_abs = {}
top_10_importances_all_subjects_negative = {}
for subject_index in all_subject_importances.keys():
        top_10_importances_all_subjects[subject_index] = get_top_k_keys(all_subject_importances[subject_index], 10)
        top_10_importances_all_subjects_abs[subject_index] = get_top_k_keys(all_subject_importances_abs[subject_index], 10)
        top_10_importances_all_subjects_negative[subject_index] = get_top_k_keys(all_subject_importances[subject_index], 10, reverse=False)

In [ ]:
import itertools
cfg = load_config()
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    top1_abs = top_10_importances_all_subjects_abs[subject_index1]
    top2_abs = top_10_importances_all_subjects_abs[subject_index2]
    top1 = top_10_importances_all_subjects[subject_index1]
    top2 = top_10_importances_all_subjects[subject_index2]
    top1_negative = top_10_importances_all_subjects_negative[subject_index1]
    top2_negative = top_10_importances_all_subjects_negative[subject_index]
    info_subj1 = load_subject_info(subject_index1)

    agreement = calculate_agreement_top_channels(top1, top2)
    agreement_abs = calculate_agreement_top_channels(top1_abs, top2_abs)
    agreement_negative = calculate_agreement_top_channels(top1_negative, top2_negative)
    #print(agreement)
    plot_agreement_topomap(top1, top2, top1_negative, top2_negative, top1_abs, top2_abs, info_subj1.ch_names, info_subj1, agreement, agreement_negative, agreement_abs, subject_index1, subject_index2)

## clustering

In [ ]:
import itertools
def plot_cluster_topoplots(linkage_matrix, top_all_subjects, subjects, all_subjects_info, common_channels, n_clusters=5, label="positive"):
    """
    Plot topographic maps of cluster centroids for each cluster.
    
    Parameters:
    -----------
    linkage_matrix : np.array
        Linkage matrix from hierarchical clustering
    subjects : list
        List of subject IDs
    all_subjects_data : dict
        Dictionary containing subject data
    common_channels : list
        List of common channel names
    n_clusters : int
        Number of clusters to extract
    """
    # Extract cluster labels
    cluster_labels = get_cluster_labels(linkage_matrix, n_clusters)
    np.save(f"cluster_labels_z_normalized_{label}.npy", cluster_labels)
    
    # Create figure
    fig, axs = plt.subplots(1, n_clusters, figsize=(15, 5))
    ch_names = all_subjects_info[1]["ch_names"]
    # Iterate through each cluster
    for cluster_id in range(1, n_clusters + 1):
        # Find subjects in this cluster
        cluster_subjects = [subj for subj, label in zip(subjects, cluster_labels) if label == cluster_id]
        
        # Average the top channels for these subjects
        cluster_top_channels = []
        all_agree_channels_cluster = np.zeros(len(ch_names))
        for subject_id1, subject_id2 in itertools.combinations(cluster_subjects,2):
            #cluster_top_channels.extend(filtered_top_channels_median_all_subjects[subject_id1][band_name][factor])
            pair_agreement = np.intersect1d(top_all_subjects[subject_id1], 
                                                  top_all_subjects[subject_id2])
            pair_agreement = [ch for ch in pair_agreement if ch in common_channels]
            for ch in pair_agreement:
                all_agree_channels_cluster[ch_names.index(ch)] += 1

            # find all channels that are in the top 
            #for band_name in filtered_top_channels_median_all_subjects[subject_id]:
            #    cluster_top_channels.extend(filtered_top_channels_median_all_subjects[subject_id][band_name][factor])
        # Compute average topographic map
        #cluster_topo = np.zeros(len(ch_names))

        # from ch_names of a subject find indices of common_channels+
        # then set the corresponding indices to 
        #for ch in cluster_top_channels:
            #cluster_topo[ch_names.index(ch)] = 1



        # Plot topomap
        mne.viz.plot_topomap(all_agree_channels_cluster, all_subjects_info[1]["info_subj"], show=False, names=ch_names, axes=axs[cluster_id - 1])
        axs[cluster_id - 1].set_title(f'Cluster {cluster_id}, N: {len(cluster_subjects)}', fontsize=18)
    
    plt.suptitle('Average topomap of clusters', fontsize=20)
    plt.tight_layout()
    fig.savefig(f"z_normalized_top_k_topomap_{label}.png")
    plt.show()

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_explanation_function(top_10_importances_all_subjects, all_subjects_info, common_channels, sigma=0.0025, ranking_method="normal")

In [ ]:
plot_cluster_topoplots(linkage_matrix, top_10_importances_all_subjects, subjects, all_subjects_info, common_channels, n_clusters=5, label="normal")

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_explanation_function(top_10_importances_all_subjects_negative, all_subjects_info, common_channels, sigma=0.0025, ranking_method="normal")

In [ ]:
plot_cluster_topoplots(linkage_matrix, top_10_importances_all_subjects_negative, subjects, all_subjects_info, common_channels, n_clusters=5, label="negative")

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_explanation_function(top_10_importances_all_subjects_abs, all_subjects_info, common_channels, sigma=0.0025, ranking_method="normal")

In [ ]:
plot_cluster_topoplots(linkage_matrix, top_10_importances_all_subjects_abs, subjects, all_subjects_info, common_channels, n_clusters=5, label="abs")

## rank correlations

In [ ]:
all_subject_importances = get_all_subject_importanes()
all_subject_importances_abs = {}
for subject_index, importance in all_subject_importances.items():
    all_subject_importances_abs[subject_index] = {ch: abs(importance[ch]) for ch in importance.keys()}

In [ ]:
import scipy
def calculate_pairwise_correlations(importances1, importances2, common_channels):

    correlation_results = {}

        # Extract the top 10 channels for both subjects

            # Extract values for common channels only
    common_channel_values1 = np.array([importances1[ch] for ch in common_channels])
    common_channel_values2 = np.array([importances2[ch] for ch in common_channels])
            
    # Calculate correlations
    pearson_corr = np.corrcoef(common_channel_values1, common_channel_values2)[0,1]
    spearman_corr = scipy.stats.spearmanr(common_channel_values1, common_channel_values2).correlation
            
    # Store results
    correlation_results= {
                'pearson': pearson_corr,
                'spearman': spearman_corr}
  
    return correlation_results

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, take_abs=False, label="positive"):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman'])

        else:

            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman']
        


        # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    np.save(f"distance_matrices/top_k_normalized_{label}_pearson.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/top_k_normalized_{label}_spearman.npy", distance_matrix_spearman)


    
    fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

    axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
    axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

    #cbar = fig.colorbar(im_spearman, ax=axs.ravel().tolist(), location='bottom', shrink=0.35)
    #cbar.set_label('Correlation')

    # Set axis labels
    for ax in axs:
        ax.set_xticks(np.arange(len(subject_indices)))
        ax.set_yticks(np.arange(len(subject_indices)))
        ax.set_xticklabels(subject_indices)
        ax.set_yticklabels(subject_indices)
        ax.set_xlabel('Subject indices')
        ax.set_ylabel('Subject indices')
        plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
    
    axs[0].set_title('Pearson correlation')
    axs[1].set_title('Spearman correlation')

    # Add text annotations
    for i in range(len(subject_indices)):
        for j in range(len(subject_indices)):
            axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                       ha="center", va="center", color="white")
            axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                       ha="center", va="center", color="white")



                                           

In [ ]:
rank_correlations_all_pairs_abs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    rank_correlations_all_pairs_abs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(all_subject_importances_abs[subject_index1], all_subject_importances_abs[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs_abs, take_abs=False, label="abs")

In [ ]:
rank_correlations_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    rank_correlations_all_pairs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(all_subject_importances[subject_index1], all_subject_importances[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs, take_abs=False, label="positive")